In [ ]:

# ============================================================
# Environment + Imports
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imagecodecs"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os, time, gc, warnings, json, random, math, traceback, hashlib, shutil
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt_fn
from torch.utils.data import Dataset, DataLoader
from collections import OrderedDict

import scipy.ndimage as ndi
from scipy.ndimage import (distance_transform_edt, maximum_filter,
                           generate_binary_structure, label as cc_label,
                           binary_erosion, binary_dilation, binary_opening)

try:
    import tifffile
    def read_tif(path):
        return tifffile.imread(path)
except ImportError:
    from PIL import Image
    def read_tif(path):
        img = Image.open(path)
        frames = []
        try:
            while True:
                frames.append(np.array(img))
                img.seek(img.tell() + 1)
        except EOFError:
            pass
        if len(frames) == 0:
            raise RuntimeError(f"Empty TIF: {path}")
        return np.stack(frames, axis=0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENV] PyTorch {torch.__version__}, Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1e9
    print(f"[ENV] GPU: {props.name}, VRAM: {VRAM_GB:.1f} GB")
else:
    VRAM_GB = 0

T_START = time.time()
def elapsed_h():
    return (time.time() - T_START) / 3600.0
def budget_ok(max_h):
    return elapsed_h() < max_h

# v2.5: TF32 + cudnn benchmark for free speed on Ampere+ GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("[ENV] TF32 + cudnn.benchmark enabled")

# v2.5: Resource logging (RAM + GPU)
def log_resources(tag=""):
    try:
        import psutil
        ram = psutil.virtual_memory()
        msg = f"[RES {tag}] RAM={ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB ({ram.percent}%)"
    except ImportError:
        msg = f"[RES {tag}]"
    if DEVICE.type == "cuda":
        alloc = torch.cuda.memory_allocated() / 1e9
        resv = torch.cuda.memory_reserved() / 1e9
        msg += f" GPU={alloc:.1f}/{resv:.1f}GB alloc/resv"
    print(msg, flush=True)

# v2.5: Worker init for DataLoader determinism (used if num_workers > 0)
def worker_init_fn(worker_id):
    seed_val = 42 + worker_id
    np.random.seed(seed_val)
    random.seed(seed_val)

log_resources("startup")


In [ ]:

# ============================================================
# Configuration - MODEL_C (role: connectivity)
# ============================================================
MODEL_NAME   = "model_c"
MODEL_ROLE   = "connectivity"  # v2.3: ensemble role (structure/boundary/connectivity)
SEED         = 2024
FOLD_IDX     = 2
NUM_FOLDS    = 5

# Architecture (nnUNet v2 3d_fullres plan)
NUM_CLASSES  = 2
IGNORE_LABEL = 255
IN_CHANNELS  = 1
FEATURES     = [32, 64, 128, 256, 320, 320]
BLOCKS       = [1, 3, 4, 6, 6, 6]
STRIDES      = [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]]
DEC_CONVS    = [1, 1, 1, 1, 1]
PATCH_SIZE       = (160, 160, 160)
PATCH_SIZE_EARLY = (min(128, 160), min(128, 160), min(128, 160))  # Fast learning phase
POLISH_FRAC      = 0.75  # Switch to full PATCH_SIZE at this fraction of time budget

# Training
BATCH_SIZE       = 1
ITERS_PER_EPOCH  = 250
EPOCHS_BUDGET    = 300
INITIAL_LR       = 0.003
WEIGHT_DECAY     = 3e-5
MOMENTUM         = 0.99
MAX_TRAIN_HOURS  = 7.5
GRAD_CKPT        = True
GRAD_CLIP        = 1.0  # v2.3: per-model gradient clipping norm

# v2.2: Cosine + warmup LR schedule
WARMUP_EPOCHS    = 5
LR_FLOOR         = 1e-06

# v2.2: Gradient accumulation (effective BS = BATCH_SIZE * GRAD_ACCUM)
GRAD_ACCUM       = 2

# v2.2: EMA
EMA_DECAY        = 0.999

# v2.2: Label smoothing
LABEL_SMOOTH     = 0.02

# Loss weights
W_CE   = 1.0
W_DICE = 1.0
W_MSR  = 0.6
W_BND  = 0.3
W_TOPO = 0.35    # topology bridge penalty (anti-merge, anti-handle)

# Deep supervision weights (nnUNet style: last=0)
DS_WEIGHTS_RAW = [1.0, 0.5, 0.25, 0.125, 0.0]
_s = sum(DS_WEIGHTS_RAW)
DS_WEIGHTS = [w / _s for w in DS_WEIGHTS_RAW]

# v2.3: Polish phase tuning (safe transitions, no detonation)
POLISH_TOPO_SCALE = 1.3  # topo multiplier in polish (1.3 = gradual, not 2.0)
POLISH_DS_SCALE   = 0.5    # DS multiplier in polish (0.5 = keep some coarse signal)

# Augmentation (scaled per model for diversity)
AUG_FLIP_PROB     = 0.5
AUG_ROT90_PROB    = 0.3
AUG_NOISE_PROB    = 0.1
AUG_NOISE_STD     = 0.07
AUG_BRIGHT_PROB   = 0.2
AUG_BRIGHT_RANGE  = 0.08
AUG_CONTRAST_PROB = 0.2
AUG_GAMMA_PROB    = 0.15

# Sampling (v2.3: per-model tuning for role diversity)
P_BOUNDARY = 0.35
P_RING     = 0.12      # near-FG ring negatives
P_FG       = 0.33
P_BG       = 0.15
P_HARD     = 0.05      # hard-example mining probability
HARD_CACHE_SIZE = 512
BOUNDARY_DILATE = 3
RING_WIDTH      = 3    # dilation width for ring negatives
IGNORE_REJECT   = 0.7 # reject patches with ignore fraction above this
MAX_REJECT_ATTEMPTS = 8
N_EPOCH_VOLS    = 8  # volumes per active window (cache-friendly)

# Validation
VAL_EVERY   = 8
VAL_N       = 3
VAL_OVERLAP = 0.25

# Paths
ROOT_CANDS = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = next((p for p in ROOT_CANDS if os.path.exists(p)), None)
if ROOT_DIR is None:
    print("[WARN] Competition data not found. Using placeholder.")
    ROOT_DIR = "/kaggle/input/competitions/vesuvius-challenge-surface-detection"
TRAIN_IMG_DIR = os.path.join(ROOT_DIR, "train_images")
TRAIN_LBL_DIR = os.path.join(ROOT_DIR, "train_labels")
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[CFG] {MODEL_NAME} ({MODEL_ROLE}): seed={SEED}, fold={FOLD_IDX}, patch={PATCH_SIZE}")
print(f"[CFG] Loss: CE={W_CE}, Dice={W_DICE}, MSR={W_MSR}, BND={W_BND}, TOPO={W_TOPO}, smooth={LABEL_SMOOTH}")
print(f"[CFG] Training: LR={INITIAL_LR}, clip={GRAD_CLIP}, warmup={WARMUP_EPOCHS}, accum={GRAD_ACCUM}, EMA={EMA_DECAY}")
print(f"[CFG] Sampling: bnd={P_BOUNDARY}, ring={P_RING}, fg={P_FG}, bg={P_BG}, hard={P_HARD}")
print(f"[CFG] Polish: topo_scale={POLISH_TOPO_SCALE}, ds_scale={POLISH_DS_SCALE}")


In [ ]:

# ============================================================
# Data Discovery + Train/Val Split
# ============================================================
vol_infos = []
if os.path.isdir(TRAIN_IMG_DIR):
    for f in sorted(os.listdir(TRAIN_IMG_DIR)):
        if f.endswith(".tif"):
            vid = f.replace(".tif", "")
            img_p = os.path.join(TRAIN_IMG_DIR, f)
            lbl_p = os.path.join(TRAIN_LBL_DIR, f)
            if os.path.exists(lbl_p):
                vol_infos.append({"id": vid, "image": img_p, "label": lbl_p})

print(f"[DATA] Found {len(vol_infos)} labeled volumes")

# Deterministic fold split (fixed seed=42 across all models for consistent folds)
_rng = random.Random(42)
_idx = list(range(len(vol_infos)))
_rng.shuffle(_idx)
_fold_sz = max(1, len(_idx) // NUM_FOLDS)
_val_start = FOLD_IDX * _fold_sz
_val_end = min(_val_start + _fold_sz, len(_idx))
_val_set = set(_idx[_val_start:_val_end])

train_vols = [v for i, v in enumerate(vol_infos) if i not in _val_set]
val_vols   = [v for i, v in enumerate(vol_infos) if i in _val_set]

print(f"[DATA] Fold {FOLD_IDX}: {len(train_vols)} train, {len(val_vols)} val")
for v in train_vols:
    print(f"  Train: {v['id']}")
for v in val_vols:
    print(f"  Val:   {v['id']}")


In [ ]:

# ============================================================
# ResidualEncoderUNet - matches nnUNet v2 3d_fullres plan
# Features: [32, 64, 128, 256, 320, 320], Blocks: [1,3,4,6,6,6]
# InstanceNorm3d + LeakyReLU, ~111M params
# ============================================================

class ConvBlock3D(nn.Module):
    """Conv3d -> InstanceNorm3d -> LeakyReLU"""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, bias=True):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size] * 3
        if isinstance(stride, int):
            stride = [stride] * 3
        padding = [k // 2 for k in kernel_size]
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size, stride=stride,
                              padding=padding, bias=bias)
        self.norm = nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class ResBlock3D(nn.Module):
    """Residual: conv->norm->act->conv->norm + skip -> act"""
    def __init__(self, ch, kernel_size=3, bias=True):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm1 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm2 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)

    def forward(self, x):
        r = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + r)


class ResidualEncoderUNet(nn.Module):
    """
    nnUNet v2 ResidualEncoderUNet for 3D segmentation.
    - Encoder: strided conv + residual blocks per stage
    - Decoder: ConvTranspose + concat skip + conv blocks
    - Deep supervision: seg heads at each decoder resolution
    """
    def __init__(self, in_channels=1, num_classes=2,
                 features=(32, 64, 128, 256, 320, 320),
                 blocks=(1, 3, 4, 6, 6, 6),
                 strides=((1,1,1),(2,2,2),(2,2,2),(2,2,2),(2,2,2),(2,2,2)),
                 dec_convs=(1, 1, 1, 1, 1),
                 deep_supervision=True, use_grad_ckpt=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self._ckpt = use_grad_ckpt
        n = len(features)

        # Encoder
        self.enc = nn.ModuleList()
        for s in range(n):
            in_c = in_channels if s == 0 else features[s - 1]
            out_c = features[s]
            st = list(strides[s]) if isinstance(strides[s], (list, tuple)) else [strides[s]] * 3
            layers = [ConvBlock3D(in_c, out_c, 3, stride=st)]
            for _ in range(blocks[s] - 1):
                layers.append(ResBlock3D(out_c, 3))
            self.enc.append(nn.Sequential(*layers))

        # Decoder
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        self.seg = nn.ModuleList()
        for i in range(n - 1):
            s = n - 1 - i
            enc_ch = features[s]
            skip_ch = features[s - 1]
            out_ch = features[s - 1]
            st = list(strides[s]) if isinstance(strides[s], (list, tuple)) else [strides[s]] * 3
            self.up.append(
                nn.ConvTranspose3d(enc_ch, enc_ch, kernel_size=st, stride=st, bias=True)
            )
            n_convs = dec_convs[i] if i < len(dec_convs) else 1
            dec_layers = []
            for c in range(n_convs):
                ic = (enc_ch + skip_ch) if c == 0 else out_ch
                dec_layers.append(ConvBlock3D(ic, out_ch, 3))
            self.dec.append(nn.Sequential(*dec_layers))
            self.seg.append(nn.Conv3d(out_ch, num_classes, 1))

    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            if self._ckpt and self.training and i > 0:
                x = grad_ckpt_fn(enc, x, use_reentrant=False)
            else:
                x = enc(x)
            skips.append(x)

        outputs = []
        x = skips[-1]
        for i, (up_m, dec_m, seg_m) in enumerate(zip(self.up, self.dec, self.seg)):
            skip = skips[-(i + 2)]
            x = up_m(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
            if self._ckpt and self.training:
                x = grad_ckpt_fn(dec_m, x, use_reentrant=False)
            else:
                x = dec_m(x)
            outputs.append(seg_m(x))

        outputs = outputs[::-1]  # [finest, ..., coarsest]

        if self.deep_supervision and self.training:
            return outputs
        return outputs[0]


# Build model
model = ResidualEncoderUNet(
    in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
    features=FEATURES, blocks=BLOCKS, strides=STRIDES,
    dec_convs=DEC_CONVS, deep_supervision=True, use_grad_ckpt=GRAD_CKPT
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"[MODEL] ResidualEncoderUNet: {n_params:.1f}M params")

# Model hash for reproducibility
_h = hashlib.sha1()
for k, v in sorted(model.state_dict().items()):
    _h.update(k.encode())
    _h.update(str(v.shape).encode())
MODEL_HASH = _h.hexdigest()[:12]
print(f"[MODEL] Architecture hash: {MODEL_HASH}")


In [ ]:

# ============================================================
# Loss Functions: CE + BoundaryWeightedCE + Dice + MedialSurfaceRecall
# + Deep Supervision wrapper + label smoothing + per-component logging
# ============================================================

def compute_boundary_weight_map(target, ignore_label=IGNORE_LABEL, dilate=2, weight=3.0):
    """Per-voxel weight map that upweights boundary regions."""
    B = target.shape[0]
    wmap = torch.ones_like(target, dtype=torch.float32)
    for b in range(B):
        tgt_np = target[b].cpu().numpy()
        fg = (tgt_np == 1).astype(np.uint8)
        if fg.sum() == 0:
            continue
        dilated = ndi.binary_dilation(fg, iterations=dilate).astype(np.uint8)
        eroded = ndi.binary_erosion(fg, iterations=dilate).astype(np.uint8)
        boundary = ((dilated - eroded) > 0).astype(np.float32)
        wmap[b] += torch.from_numpy(boundary * (weight - 1.0)).to(target.device)
    wmap[target == ignore_label] = 0.0
    return wmap


def soft_dice_loss(logits, target, ignore_label=IGNORE_LABEL, eps=1e-5):
    """Per-sample soft Dice loss, ignoring IGNORE_LABEL voxels."""
    C = logits.shape[1]
    # v2.4: Clamp logits + force float32 before softmax (AMP safety)
    logits = logits.float().clamp(-50, 50)
    probs = torch.softmax(logits, dim=1)
    valid = (target != ignore_label)
    tgt = target.clone()
    tgt[~valid] = 0
    onehot = F.one_hot(tgt.long(), num_classes=C).permute(0, 4, 1, 2, 3).float()
    mask = valid.unsqueeze(1).float()
    probs = probs * mask
    onehot = onehot * mask
    dims = (0, 2, 3, 4)
    inter = (probs * onehot).sum(dims)
    denom = probs.sum(dims) + onehot.sum(dims)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def boundary_weighted_ce(logits, target, boundary_wmap, ignore_label=IGNORE_LABEL):
    """CE loss with per-voxel boundary weighting."""
    # v2.4b: FP32 with autocast disabled (custom reduction is fragile under AMP)
    with torch.amp.autocast("cuda", enabled=False):
        logits = logits.float().clamp(-20, 20)  # Tighter clamp for boundary stability
        ce_per_voxel = F.cross_entropy(logits, target, ignore_index=ignore_label, reduction='none')
        weighted = ce_per_voxel * boundary_wmap.float()
        valid = (target != ignore_label).float()
        n_valid = valid.sum() + 1e-8
    return weighted.sum() / n_valid


def medial_surface_recall_loss(fg_probs, medial_mask, valid_mask):
    """Penalize missing medial surface voxels in prediction."""
    # v2.4b: FP32 with autocast disabled for numerical stability
    with torch.amp.autocast("cuda", enabled=False):
        fg_probs = fg_probs.float().clamp(1e-4, 1 - 1e-4)
        med = medial_mask.float() * valid_mask.float()
        n_med = med.sum() + 1e-8
        recall = (fg_probs * med).sum() / n_med
    return 1.0 - recall


def topo_bridge_penalty(logits, target, kernel=3, ignore_label=IGNORE_LABEL):
    """Differentiable penalty on thin bridge-like structures.
    Morphological opening (erosion→dilation) removes thin connections.
    Penalize prediction confidence at locations opening would remove.
    Directly targets VOI_merge and TopoScore (handles/tunnels)."""
    # v2.4b: Entire topo computation in FP32 with autocast DISABLED.
    # Just .float() on logits isn't enough — pooling ops can still autocast to fp16.
    with torch.amp.autocast("cuda", enabled=False):
        logits = logits.float().clamp(-20, 20)  # Tighter clamp for topo stability
        fg_prob = torch.softmax(logits, dim=1)[:, 1]  # (B, D, H, W)
        p5 = fg_prob.unsqueeze(1)  # (B, 1, D, H, W) for pooling
        pad = kernel // 2
        # Morphological opening: erode (min-pool) then dilate (max-pool)
        eroded = -F.max_pool3d(-p5, kernel, stride=1, padding=pad)
        opened = F.max_pool3d(eroded, kernel, stride=1, padding=pad)
        opened = opened.squeeze(1)  # (B, D, H, W)
        # Thin structures = what opening removed (detached mask for stability)
        thin_mask = (fg_prob - opened).clamp(min=0).detach()
        # Only penalize at valid (non-ignore) locations
        valid = (target != ignore_label).float()
        penalty = (fg_prob * thin_mask * valid).sum() / (valid.sum() + 1e-8)
    return penalty


def compute_loss(outputs, target, medial):
    """
    Deep supervision loss with per-component tracking.
    Returns (total_loss, components_dict).
    """
    total = torch.tensor(0.0, device=target.device)
    comps = {"ce": 0.0, "dice": 0.0, "msr": 0.0, "bnd": 0.0, "topo": 0.0}

    # Precompute boundary weight map at full resolution
    bnd_wmap_full = compute_boundary_weight_map(target) if W_BND > 0 else None

    for i, (logits, w) in enumerate(zip(outputs, DS_WEIGHTS)):
        # Late-phase: reduce deep supervision (focus compute on main output)
        if i > 0:
            w = w * _DS_SCALE
        if w < 1e-6:
            continue
        # Downsample target to match logits resolution
        if logits.shape[2:] != target.shape[1:]:
            tgt = F.interpolate(
                target.float().unsqueeze(1),
                size=logits.shape[2:], mode="nearest"
            ).long().squeeze(1)
        else:
            tgt = target

        # CE with label smoothing (v2.4: float32 + clamp for AMP safety)
        logits_c = logits.float().clamp(-50, 50)
        ce = F.cross_entropy(logits_c, tgt, ignore_index=IGNORE_LABEL,
                             label_smoothing=LABEL_SMOOTH)
        ce = torch.nan_to_num(ce, nan=0.0, posinf=10.0, neginf=0.0)
        # Dice (soft_dice_loss does its own clamping)
        dc = soft_dice_loss(logits, tgt, IGNORE_LABEL)
        dc = torch.nan_to_num(dc, nan=0.0, posinf=10.0, neginf=0.0)
        total = total + w * (W_CE * ce + W_DICE * dc)
        # v2.4: Guard against NaN in individual loss terms
        if not torch.isfinite(total):
            return total, comps  # Early exit, NaN guard in training loop will handle
        comps["ce"] += w * W_CE * ce.item()
        comps["dice"] += w * W_DICE * dc.item()

        # Boundary-weighted CE at full resolution only
        if W_BND > 0 and i == 0 and bnd_wmap_full is not None:
            bce = boundary_weighted_ce(logits, tgt, bnd_wmap_full)
            bce = torch.nan_to_num(bce, nan=0.0, posinf=10.0, neginf=0.0)
            total = total + w * W_BND * bce
            comps["bnd"] += w * W_BND * bce.item()

    # MedialSurfaceRecall only at full resolution; skip if medial is empty (scheduled skip)
    # v2.4: ramped by _TOPO_RAMP (0→1 over first 15% of training) to prevent early NaN
    if W_MSR > 0 and len(outputs) > 0 and medial.sum() > 0 and _TOPO_RAMP > 0:
        fg_probs = torch.softmax(outputs[0].float(), dim=1)[:, 1]
        valid = (target != IGNORE_LABEL)
        msr = medial_surface_recall_loss(fg_probs, medial, valid)
        msr = torch.nan_to_num(msr, nan=0.0, posinf=10.0, neginf=0.0)
        msr_w = W_MSR * _TOPO_RAMP
        total = total + msr_w * msr
        comps["msr"] += msr_w * msr.item()

    # v2.2: Topology bridge penalty (anti-merge, anti-handle) — scaled in polish phase
    # v2.4: also ramped by _TOPO_RAMP to prevent slamming at full weight early
    if W_TOPO > 0 and len(outputs) > 0 and _TOPO_RAMP > 0:
        tp = topo_bridge_penalty(outputs[0], target)
        tp = torch.nan_to_num(tp, nan=0.0, posinf=10.0, neginf=0.0)
        topo_w = W_TOPO * _TOPO_SCALE * _TOPO_RAMP
        total = total + topo_w * tp
        comps["topo"] += topo_w * tp.item()

    return total, comps

print("[LOSS] BndCE + Dice + MSR + TopoBridge + label_smooth={:.3f} with deep supervision".format(LABEL_SMOOTH))


In [ ]:

# ============================================================
# Dataset + Augmentation
# v2.2: ring negatives, patch rejection, volume cycling
# ============================================================

def normalize_volume(vol):
    """Z-score normalization (matching nnUNet use_mask_for_norm=false)."""
    v = vol.astype(np.float32)
    mu = v.mean()
    sd = v.std() + 1e-8
    return (v - mu) / sd


def compute_medial_surface(mask_3d):
    """Approximate medial surface via distance transform local maxima. ~0.05s for 192^3."""
    if mask_3d.sum() == 0:
        return np.zeros_like(mask_3d, dtype=np.float32)
    dt = distance_transform_edt(mask_3d.astype(bool))
    lm = maximum_filter(dt, size=3)
    medial = ((dt > 0) & (dt >= lm)).astype(np.float32)
    return medial


_ELASTIC_P   = 0.15  # Module-level: updated by set_epoch for schedule
_MEDIAL_P    = 1.0   # Module-level: fraction of patches that compute medial
_DS_SCALE    = 1.0   # Module-level: deep supervision weight multiplier (0=disabled)
_TOPO_SCALE  = 1.0   # Module-level: topology penalty multiplier (2x in polish phase)
_TOPO_RAMP   = 0.0   # Module-level: ramp multiplier for topo/MSR (0→1 over RAMP_FRAC)
_PHASE       = "learn"  # "learn" or "polish"
TOPO_RAMP_FRAC = 0.15  # Fraction of training time to ramp topo/MSR from 0→1

# v2.5: OOM degradation state (survives across epochs)
_OOM_SHRINK     = 0   # Patch shrink level (each level = -16 per dim)
_OOM_THIS_EPOCH = 0   # OOM count in current epoch

def elastic_deform_3d(img, lbl, med, alpha=6.0, grid_step=32):
    """Elastic deformation via coarse random displacement grid + spline interpolation.
    Teaches surface stability under warp -> helps SurfaceDice.
    Uses coarse grid for speed (~0.3s for 192^3 vs ~2s for dense).
    Probability is scheduled: high early (invariance), low late (precision)."""
    if random.random() > _ELASTIC_P:
        return img, lbl, med
    shape = img.shape
    # Coarse displacement grid
    cshape = tuple(max(2, s // grid_step + 1) for s in shape)
    rng_state = np.random.RandomState()
    dz = rng_state.randn(*cshape).astype(np.float32) * alpha
    dy = rng_state.randn(*cshape).astype(np.float32) * alpha
    dx = rng_state.randn(*cshape).astype(np.float32) * alpha
    # Upsample to full resolution via spline zoom
    zoom_f = tuple(s / c for s, c in zip(shape, cshape))
    dz = ndi.zoom(dz, zoom_f, order=3)[:shape[0], :shape[1], :shape[2]]
    dy = ndi.zoom(dy, zoom_f, order=3)[:shape[0], :shape[1], :shape[2]]
    dx = ndi.zoom(dx, zoom_f, order=3)[:shape[0], :shape[1], :shape[2]]
    # Build coordinate grids + displacement
    z, y, x = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]),
                           np.arange(shape[2]), indexing='ij')
    coords = [z + dz, y + dy, x + dx]
    # Apply: linear interp for image, nearest for labels & medial
    img = ndi.map_coordinates(img, coords, order=1, mode='reflect').astype(np.float32)
    lbl = ndi.map_coordinates(lbl, coords, order=0, mode='constant',
                              cval=IGNORE_LABEL).astype(np.uint8)
    med = ndi.map_coordinates(med, coords, order=0, mode='constant',
                              cval=0).astype(np.float32)
    return img, lbl, med


def augment_patch(img, lbl, med):
    """Numpy augmentation for a 3D patch. Intensity aug scaled by AUG_SCALE."""
    # v2.2: Elastic deformation (surface stability -> SurfaceDice)
    img, lbl, med = elastic_deform_3d(img, lbl, med)

    # Random flips
    for ax in range(3):
        if random.random() < AUG_FLIP_PROB:
            img = np.flip(img, axis=ax).copy()
            lbl = np.flip(lbl, axis=ax).copy()
            med = np.flip(med, axis=ax).copy()

    # Random 90-degree rotation in XY plane
    if random.random() < AUG_ROT90_PROB:
        k = random.choice([1, 2, 3])
        img = np.rot90(img, k, axes=(1, 2)).copy()
        lbl = np.rot90(lbl, k, axes=(1, 2)).copy()
        med = np.rot90(med, k, axes=(1, 2)).copy()

    # Intensity augmentations
    if random.random() < AUG_BRIGHT_PROB:
        img = img + np.random.uniform(-AUG_BRIGHT_RANGE, AUG_BRIGHT_RANGE)
    if random.random() < AUG_CONTRAST_PROB:
        img = img * np.random.uniform(0.93, 1.07)
    if random.random() < AUG_GAMMA_PROB:
        mn = img.min()
        rng = img.max() - mn + 1e-8
        img_01 = (img - mn) / rng
        gamma = np.exp(np.random.uniform(-0.2, 0.3))
        img = np.power(np.clip(img_01, 1e-8, None), gamma) * rng + mn
    if random.random() < AUG_NOISE_PROB:
        img = img + np.random.normal(0, AUG_NOISE_STD, img.shape).astype(np.float32)

    # v2.2: Random cuboid cutout (forces continuity inference -> fewer splits)
    if random.random() < 0.12:
        cs = [random.randint(s // 8, s // 4) for s in img.shape]
        cz = random.randint(0, img.shape[0] - cs[0])
        cy = random.randint(0, img.shape[1] - cs[1])
        cx = random.randint(0, img.shape[2] - cs[2])
        img[cz:cz+cs[0], cy:cy+cs[1], cx:cx+cs[2]] = 0.0
        # Don't modify label: forces model to infer structure from context

    return img, lbl, med


class _LRU:
    """Simple LRU cache."""
    def __init__(self, cap=2):
        self.cap = cap
        self.od = OrderedDict()
    def get(self, key):
        if key in self.od:
            self.od.move_to_end(key)
            return self.od[key]
        return None
    def put(self, key, val):
        self.od[key] = val
        self.od.move_to_end(key)
        while len(self.od) > self.cap:
            self.od.popitem(last=False)


class VesuviusPatchDataset(Dataset):
    """
    Random patch sampling with boundary/ring/FG/BG weighting.
    v2.2: ring negatives, patch rejection, volume cycling, hard mining.
    Coverage queue guarantees all volumes get seen over time.
    Overlapping swaps keep ~62% of active set (cache stays warm).
    Time-based swaps keep correlation window consistent in wall-clock.
    """
    SWAP_KEEP_FRAC = 0.62  # Fraction of active set to keep on each swap
    SWAP_INTERVAL_SEC = 90 # Time-based swap cadence (seconds)

    def __init__(self, vol_infos, patch_size, num_iters, augment=True):
        self.vols = list(vol_infos)
        self.ps = tuple(patch_size)
        self.num_iters = num_iters
        self.augment = augment
        # v2.5: RAM safety — limit volume cache (each vol ~30-100MB in memory)
        # 5 volumes keeps active-set cycling warm without risking kernel death
        max_k = min(N_EPOCH_VOLS * 2, len(self.vols))
        self.vol_cache = _LRU(5)
        self.coord_cache = _LRU(max(max_k + 4, 12))  # Coords are small, keep more
        # Hard-example mining
        self.hard_coords = []
        # Coverage queue: shuffled list of ALL volume indices, consumed K at a time.
        # When exhausted, reshuffle. Guarantees every volume gets seen.
        self._epoch = 0
        self._vol_queue = []          # will be filled on first set_epoch
        self._queue_seed_ctr = 0      # increments each reshuffle for determinism
        self._current_k = min(N_EPOCH_VOLS, len(self.vols))
        self._active_set = list(range(self._current_k))
        self._swaps_this_epoch = 0
        self._last_swap_time = time.time()

    def __len__(self):
        return self.num_iters

    def _refill_queue(self):
        """Reshuffle all volume indices into the queue (coverage guarantee)."""
        rng = random.Random(SEED + self._queue_seed_ctr)
        self._queue_seed_ctr += 1
        q = list(range(len(self.vols)))
        rng.shuffle(q)
        self._vol_queue = q

    def _next_active_set(self):
        """Overlapping swap: keep ~62% of current set, replace rest from queue.
        Keeps cache warm while steadily injecting diversity."""
        k = self._current_k
        n_keep = max(1, int(k * self.SWAP_KEEP_FRAC))
        n_new = k - n_keep
        # Keep a random subset of current active set
        if len(self._active_set) >= n_keep:
            kept = random.sample(self._active_set, n_keep)
        else:
            kept = list(self._active_set)
            n_new = k - len(kept)
        # Pop new volumes from coverage queue
        if len(self._vol_queue) < n_new:
            self._refill_queue()
        new_vols = [self._vol_queue.pop() for _ in range(n_new)]
        self._active_set = kept + new_vols
        self._last_swap_time = time.time()

    def _prewarm(self):
        """Eagerly load active-set volumes into cache (eliminates step-time spikes).
        Call after any swap to pay IO cost upfront rather than during training steps."""
        for vi in self._active_set:
            info = self.vols[vi]
            img, lbl = self._load(info)
            self._get_coords(info, lbl)

    def set_epoch(self, epoch, elapsed_hours=0.0):
        """Time-based phase control: patch-size curriculum + topology polish.
        Learn phase (0–75% time): small patches (128³), heavy aug, learn features.
        Polish phase (75–100% time): full patches, low aug, high topo weight.
        All schedules keyed to wall-clock time, not epoch count."""
        global _ELASTIC_P, _MEDIAL_P, _DS_SCALE, _TOPO_SCALE, _TOPO_RAMP, _PHASE
        global _OOM_SHRINK
        self._epoch = epoch
        self._swaps_this_epoch = 0
        time_frac = min(1.0, elapsed_hours / MAX_TRAIN_HOURS)
        prev_phase = _PHASE

        # v2.4: Topo/MSR ramp-in (prevent NaN from slamming topo at full weight early)
        if time_frac < TOPO_RAMP_FRAC:
            _TOPO_RAMP = min(1.0, time_frac / max(TOPO_RAMP_FRAC, 1e-8))
        else:
            _TOPO_RAMP = 1.0

        # ===== PATCH-SIZE CURRICULUM (the #1 win under a time cap) =====
        if time_frac < POLISH_FRAC:
            self.ps = tuple(PATCH_SIZE_EARLY)
            _PHASE = "learn"
        else:
            self.ps = tuple(PATCH_SIZE)
            _PHASE = "polish"

        # v2.5: OOM degradation — shrink patch if OOMs accumulated
        if _OOM_SHRINK > 0:
            shrink = _OOM_SHRINK * 16
            self.ps = tuple(max(64, p - shrink) for p in self.ps)
            if _OOM_SHRINK >= 2:
                _ELASTIC_P = 0.0  # Disable elastic to save GPU memory

        # ===== VOLUME CURRICULUM: K grows over training =====
        if time_frac < 0.10:
            self._current_k = max(4, N_EPOCH_VOLS // 2)
        elif time_frac < 0.70:
            self._current_k = N_EPOCH_VOLS
        else:
            self._current_k = min(N_EPOCH_VOLS * 2, len(self.vols))
        self._current_k = min(self._current_k, len(self.vols))

        # ===== LATE-PHASE SCHEDULES =====
        if _PHASE == "learn":
            _ELASTIC_P  = 0.15    # Heavy aug for invariance
            _MEDIAL_P   = 0.5     # Save CPU (more steps matter more)
            _DS_SCALE   = 1.0     # Full deep supervision
            _TOPO_SCALE = 1.0     # Normal topology penalty
        else:  # polish
            _ELASTIC_P  = 0.03    # Minimal aug (precision matters)
            _MEDIAL_P   = 1.0     # Full medial (surface quality matters)
            _DS_SCALE   = POLISH_DS_SCALE    # v2.3: configurable (0.5 default, keeps coarse signal)
            _TOPO_SCALE = POLISH_TOPO_SCALE  # v2.3: configurable (1.3 default, gradual not 2x)

        if _PHASE != prev_phase:
            pD, pH, pW = self.ps
            print(f"\n  >>> PHASE TRANSITION: {prev_phase} -> {_PHASE} "
                  f"(t={time_frac:.1%}, patch={pD}x{pH}x{pW}, "
                  f"topo_scale={_TOPO_SCALE:.1f}, ds_scale={_DS_SCALE:.1f}, "
                  f"elastic={_ELASTIC_P:.2f})", flush=True)

        self._next_active_set()
        self._prewarm()

    def add_hard_coord(self, vol_idx, z, y, x):
        """Called by training loop to record high-loss locations."""
        self.hard_coords.append((vol_idx, int(z), int(y), int(x)))
        if len(self.hard_coords) > HARD_CACHE_SIZE:
            self.hard_coords = self.hard_coords[-HARD_CACHE_SIZE:]

    def _load(self, info):
        vid = info["id"]
        cached = self.vol_cache.get(vid)
        if cached is not None:
            return cached
        img = read_tif(info["image"])
        lbl = read_tif(info["label"])
        img = normalize_volume(img)
        lbl = lbl.astype(np.uint8)
        lbl[lbl == 2] = IGNORE_LABEL
        self.vol_cache.put(vid, (img, lbl))
        return img, lbl

    def _get_coords(self, info, lbl):
        vid = info["id"]
        cached = self.coord_cache.get(vid)
        if cached is not None:
            return cached

        fg = (lbl == 1)
        bg = (lbl == 0)

        # Boundary coords
        if fg.any() and bg.any():
            dilated = ndi.binary_dilation(fg, iterations=BOUNDARY_DILATE)
            boundary = dilated & bg
            eroded = ndi.binary_erosion(fg, iterations=BOUNDARY_DILATE)
            boundary = boundary | (fg & ~eroded)
            bnd_coords = np.argwhere(boundary)
        else:
            bnd_coords = np.empty((0, 3), dtype=np.int64)

        # v2.2: Ring negatives = dilated FG minus FG, excluding ignore
        if fg.any():
            ring_mask = ndi.binary_dilation(fg, iterations=RING_WIDTH) & ~fg & (lbl != IGNORE_LABEL)
            ring_coords = np.argwhere(ring_mask)
        else:
            ring_coords = np.empty((0, 3), dtype=np.int64)

        fg_coords = np.argwhere(fg)
        bg_coords = np.argwhere(bg)

        # Cap coord arrays for memory
        cap = 100000
        for arr_name in ['bnd_coords', 'ring_coords', 'fg_coords', 'bg_coords']:
            arr = locals()[arr_name]
            if len(arr) > cap:
                idx = np.random.choice(len(arr), cap, replace=False)
                if arr_name == 'bnd_coords': bnd_coords = arr[idx]
                elif arr_name == 'ring_coords': ring_coords = arr[idx]
                elif arr_name == 'fg_coords': fg_coords = arr[idx]
                elif arr_name == 'bg_coords': bg_coords = arr[idx]

        result = (bnd_coords, ring_coords, fg_coords, bg_coords)
        self.coord_cache.put(vid, result)
        return result

    def _sample_center(self, coords, shape):
        """Pick a random coord, clamp so patch fits within volume."""
        pD, pH, pW = self.ps
        D, H, W = shape
        if len(coords) == 0:
            z = random.randint(0, max(0, D - pD))
            y = random.randint(0, max(0, H - pH))
            x = random.randint(0, max(0, W - pW))
            return z, y, x
        idx = random.randint(0, len(coords) - 1)
        cz, cy, cx = coords[idx]
        z = max(0, min(cz - pD // 2, D - pD))
        y = max(0, min(cy - pH // 2, H - pH))
        x = max(0, min(cx - pW // 2, W - pW))
        return int(z), int(y), int(x)

    def _extract_patch(self, img, lbl, coords, shape):
        """Extract and validate a patch, with rejection for high IGNORE fraction."""
        pD, pH, pW = self.ps

        for attempt in range(MAX_REJECT_ATTEMPTS):
            if shape[0] < pD or shape[1] < pH or shape[2] < pW:
                pad_d = max(0, pD - shape[0])
                pad_h = max(0, pH - shape[1])
                pad_w = max(0, pW - shape[2])
                img_p = np.pad(img, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant')
                lbl_p = np.pad(lbl, ((0,pad_d),(0,pad_h),(0,pad_w)),
                               mode='constant', constant_values=IGNORE_LABEL)
                ip = img_p[:pD, :pH, :pW].copy()
                lp = lbl_p[:pD, :pH, :pW].copy()
                return ip, lp

            z, y, x = self._sample_center(coords, shape)
            ip = img[z:z+pD, y:y+pH, x:x+pW].copy()
            lp = lbl[z:z+pD, y:y+pH, x:x+pW].copy()

            if ip.shape != (pD, pH, pW):
                continue

            # v2.2: Patch rejection if too much IGNORE
            ign_frac = (lp == IGNORE_LABEL).sum() / lp.size
            if ign_frac <= IGNORE_REJECT:
                return ip, lp

        # Fallback: accept whatever we got
        z = random.randint(0, max(0, shape[0] - pD))
        y = random.randint(0, max(0, shape[1] - pH))
        x = random.randint(0, max(0, shape[2] - pW))
        ip = img[z:z+pD, y:y+pH, x:x+pW].copy()
        lp = lbl[z:z+pD, y:y+pH, x:x+pW].copy()
        return ip, lp

    def __getitem__(self, _idx):
        pD, pH, pW = self.ps

        # Hard mining path
        if self.hard_coords and random.random() < P_HARD:
            vi, hz, hy, hx = random.choice(self.hard_coords)
            vi = vi % len(self.vols)
            info = self.vols[vi]
            img, lbl = self._load(info)
            shape = img.shape
            z = max(0, min(hz - pD // 2, max(0, shape[0] - pD)))
            y = max(0, min(hy - pH // 2, max(0, shape[1] - pH)))
            x = max(0, min(hx - pW // 2, max(0, shape[2] - pW)))
            ip = img[z:z+pD, y:y+pH, x:x+pW].copy()
            lp = lbl[z:z+pD, y:y+pH, x:x+pW].copy()
            if ip.shape == (pD, pH, pW):
                fg_mask = (lp == 1)
                med = compute_medial_surface(fg_mask) if random.random() < _MEDIAL_P else np.zeros_like(fg_mask, dtype=np.float32)
                if self.augment:
                    ip, lp, med = augment_patch(ip, lp, med)
                return (torch.from_numpy(ip.astype(np.float32)).unsqueeze(0),
                        torch.from_numpy(lp.astype(np.int64)),
                        torch.from_numpy(med.astype(np.float32)))

        # v2.2: Time-based mid-epoch rotation (consistent correlation window)
        if _idx > 0 and (time.time() - self._last_swap_time) > self.SWAP_INTERVAL_SEC:
            self._next_active_set()
            self._prewarm()
            self._swaps_this_epoch += 1

        # v2.2: Volume cycling within active window (coverage queue)
        vol_idx = self._active_set[_idx % len(self._active_set)]
        info = self.vols[vol_idx]
        img, lbl = self._load(info)
        bnd, ring, fg, bg = self._get_coords(info, lbl)

        # Region sampling: boundary / ring / FG / BG
        r = random.random()
        if r < P_BOUNDARY and len(bnd) > 0:
            coords = bnd
        elif r < P_BOUNDARY + P_RING and len(ring) > 0:
            coords = ring
        elif r < P_BOUNDARY + P_RING + P_FG and len(fg) > 0:
            coords = fg
        elif len(bg) > 0:
            coords = bg
        else:
            coords = fg if len(fg) > 0 else bnd

        ip, lp = self._extract_patch(img, lbl, coords, img.shape)

        fg_mask = (lp == 1)
        # Medial: schedule-controlled skip to save CPU time
        if random.random() < _MEDIAL_P:
            med = compute_medial_surface(fg_mask)
        else:
            med = np.zeros_like(fg_mask, dtype=np.float32)

        if self.augment:
            ip, lp, med = augment_patch(ip, lp, med)

        img_t = torch.from_numpy(ip.astype(np.float32)).unsqueeze(0)
        lbl_t = torch.from_numpy(lp.astype(np.int64))
        med_t = torch.from_numpy(med.astype(np.float32))

        return img_t, lbl_t, med_t

print("[DATA] Dataset ready (v2.2: ring negatives, patch rejection, volume cycling)")


In [ ]:

# ============================================================
# Training Loop: v2.2
# - Cosine + warmup LR schedule
# - EMA (Exponential Moving Average) weights
# - Gradient accumulation (effective BS = BATCH_SIZE * GRAD_ACCUM)
# - NaN/Inf guard with LR backoff
# - Per-component loss logging
# - Sanity probe before training
# - Run manifest
# ============================================================

def cosine_warmup_lr(epoch, max_epoch, initial_lr, warmup=WARMUP_EPOCHS, floor=LR_FLOOR):
    """Cosine annealing with linear warmup, TIME-BASED.
    Uses elapsed wall-clock fraction so LR reaches minimum exactly when
    training hits the time budget, regardless of how many epochs fit.
    """
    time_frac = min(1.0, elapsed_h() / MAX_TRAIN_HOURS)
    warmup_frac = warmup / max(max_epoch, 1)
    if time_frac < warmup_frac:
        return max(floor, initial_lr * time_frac / max(warmup_frac, 1e-8))
    progress = (time_frac - warmup_frac) / max(1e-8, 1.0 - warmup_frac)
    return max(floor, initial_lr * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))


class EMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for k, v in model.state_dict().items():
            self.shadow[k] = v.float().clone().detach()

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in self.shadow:
                    self.shadow[k].mul_(self.decay).add_(v.float(), alpha=1 - self.decay)

    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}

    def apply(self, model):
        """Load EMA weights into model."""
        model.load_state_dict(self.shadow)

    def save(self, path):
        torch.save(self.shadow, path)


@torch.no_grad()
def val_dice_proxy(model_eval, n_vols=VAL_N):
    """Cheap patch-based validation proxy. Returns (dice, combo_proxy).
    v2.5: Uses softmax probs instead of argmax (matches inference),
    plus component-count-aware combo metric for best-by-combo checkpoint."""
    if len(val_vols) == 0:
        return 0.0, 0.0
    model_eval.eval()
    was_ds = model_eval.deep_supervision
    model_eval.deep_supervision = False

    dices = []
    combos = []
    struct26 = generate_binary_structure(3, 3)
    amp_on = (DEVICE.type == "cuda")
    n_vols = min(n_vols, len(val_vols))

    for vi in range(n_vols):
        info = val_vols[vi]
        img = normalize_volume(read_tif(info["image"]))
        lbl = read_tif(info["label"]).astype(np.uint8)
        lbl[lbl == 2] = IGNORE_LABEL

        pD, pH, pW = PATCH_SIZE
        D, H, W = img.shape
        n_patches = min(8, max(1, (D * H * W) // (pD * pH * pW)))

        for _ in range(n_patches):
            z = random.randint(0, max(0, D - pD))
            y = random.randint(0, max(0, H - pH))
            x = random.randint(0, max(0, W - pW))
            ip = img[z:z+pD, y:y+pH, x:x+pW]
            lp = lbl[z:z+pD, y:y+pH, x:x+pW]
            if ip.shape != (pD, pH, pW):
                continue
            t = torch.from_numpy(ip.astype(np.float32)[None, None]).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=amp_on):
                logits = model_eval(t)
                if isinstance(logits, (list, tuple)):
                    logits = logits[0]
            # v2.5: Use softmax probs > 0.5 instead of argmax (matches inference)
            probs = torch.softmax(logits[0].float(), dim=0)
            pred = (probs[1] > 0.5).cpu().numpy()

            valid = (lp != IGNORE_LABEL)
            if valid.sum() < 100:
                continue
            gt = (lp == 1) & valid
            pr = pred & valid
            inter = (gt & pr).sum()
            denom = gt.sum() + pr.sum()
            if denom > 0:
                d = 2.0 * inter / denom
                dices.append(d)
                # v2.5: Component-aware combo (approximates competition metric)
                try:
                    _, n_pred = cc_label(pr, structure=struct26)
                    _, n_gt = cc_label(gt, structure=struct26)
                    comp = max(0, 1.0 - abs(n_pred - n_gt) / max(n_gt, 5))
                    combos.append(0.6 * d + 0.4 * comp)
                except Exception:
                    combos.append(d)

    model_eval.deep_supervision = was_ds
    model_eval.train()
    dice = float(np.mean(dices)) if dices else 0.0
    combo = float(np.mean(combos)) if combos else 0.0
    return dice, combo


def sanity_probe(dataset, n_samples=5):
    """Pre-training sanity check: verify dataset produces valid samples."""
    print(f"\n{'='*60}")
    print(f"SANITY PROBE: Checking {n_samples} samples")
    print(f"{'='*60}")

    fg_fracs = []
    ign_fracs = []
    img_ranges = []

    for i in range(min(n_samples, len(dataset))):
        img, lbl, med = dataset[i]
        fg_frac = (lbl == 1).sum().item() / max(lbl.numel(), 1)
        ign_frac = (lbl == IGNORE_LABEL).sum().item() / max(lbl.numel(), 1)
        fg_fracs.append(fg_frac)
        ign_fracs.append(ign_frac)
        img_ranges.append((img.min().item(), img.max().item(), img.std().item()))

        print(f"  [{i}] img: shape={tuple(img.shape)}, "
              f"range=[{img.min():.3f}, {img.max():.3f}], std={img.std():.3f}")
        print(f"       lbl: FG={fg_frac*100:.1f}%, IGN={ign_frac*100:.1f}%, "
              f"medial={med.sum():.0f} vox")

    # Assertions
    assert any(f > 0 for f in fg_fracs), "FAIL: All samples have 0% FG!"
    assert all(f < 0.99 for f in ign_fracs), "FAIL: Some samples are >99% IGNORE!"
    assert all(s[2] > 0.01 for s in img_ranges), "FAIL: Image std too low!"

    print(f"\n  FG fraction: mean={np.mean(fg_fracs)*100:.1f}%, "
          f"range=[{np.min(fg_fracs)*100:.1f}%, {np.max(fg_fracs)*100:.1f}%]")
    print(f"  All sanity checks passed!")
    print(f"{'='*60}\n")


def train_model():
    global _OOM_SHRINK, _OOM_THIS_EPOCH  # v2.5: OOM degradation state
    best_loss = float("inf")
    best_val_dice = -1.0
    best_val_combo = -1.0  # v2.5: Best combo proxy (competition-metric-aligned)
    best_loss_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best_loss.pt")
    best_val_path  = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best.pt")  # PRIMARY (best dice)
    best_combo_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best_combo.pt")  # v2.5: best combo
    meta_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_meta.json")

    optimizer = torch.optim.SGD(
        model.parameters(), lr=INITIAL_LR,
        momentum=MOMENTUM, nesterov=True, weight_decay=WEIGHT_DECAY
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    # v2.2: EMA
    ema = EMA(model, decay=EMA_DECAY)

    dataset = VesuviusPatchDataset(
        train_vols, patch_size=PATCH_SIZE,
        num_iters=ITERS_PER_EPOCH, augment=True
    )
    dataloader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True, drop_last=True
    )

    # v2.2: Sanity probe
    sanity_probe(dataset, n_samples=5)

    # v2.5: Enhanced manifest (non-negotiable: everything needed to rebuild)
    manifest = {
        "version": "v2.5",
        "model": MODEL_NAME,
        "model_role": MODEL_ROLE,
        "seed": SEED,
        "fold": FOLD_IDX,
        "num_folds": NUM_FOLDS,
        "patch_size": list(PATCH_SIZE),
        "patch_size_early": list(PATCH_SIZE_EARLY),
        "features": list(FEATURES),
        "blocks": list(BLOCKS),
        # Training config
        "initial_lr": INITIAL_LR, "grad_clip": GRAD_CLIP,
        "grad_accum": GRAD_ACCUM, "ema_decay": EMA_DECAY,
        "label_smooth": LABEL_SMOOTH,
        "warmup_epochs": WARMUP_EPOCHS, "lr_floor": LR_FLOOR,
        "epochs_budget": EPOCHS_BUDGET, "max_train_hours": MAX_TRAIN_HOURS,
        # Loss weights
        "w_ce": W_CE, "w_dice": W_DICE, "w_msr": W_MSR, "w_bnd": W_BND, "w_topo": W_TOPO,
        # Sampling
        "ring_width": RING_WIDTH, "ignore_reject": IGNORE_REJECT,
        "p_boundary": P_BOUNDARY, "p_ring": P_RING, "p_fg": P_FG,
        "p_bg": P_BG, "p_hard": P_HARD,
        # Normalization + label mapping (v2.5: recorded for reproducibility)
        "normalization": "zscore",
        "label_mapping": {"bg": 0, "fg": 1, "ignore": 255, "remap_2_to_ignore": True},
        # Postproc defaults (v2.5: so submission knows what training assumed)
        "postproc_defaults": {
            "dust_min_3d": 192, "hole_max_2d": 48,
            "open_r_xy": 1, "open_r_3d": 1, "bridge_kill": True,
        },
        # Data
        "n_train": len(train_vols), "n_val": len(val_vols),
        # Environment
        "arch_hash": MODEL_HASH,
        "pytorch_version": torch.__version__,
        "device": str(DEVICE),
    }
    manifest_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"[MANIFEST] Saved: {manifest_path}")

    print(f"\n{'='*60}")
    print(f"Training {MODEL_NAME}: up to {EPOCHS_BUDGET} epochs, {ITERS_PER_EPOCH} iters/epoch")
    print(f"Effective BS={BATCH_SIZE * GRAD_ACCUM}, EMA decay={EMA_DECAY}")
    print(f"Val proxy every {VAL_EVERY} epochs on {VAL_N} volumes")
    print(f"{'='*60}")

    meta = {"model": MODEL_NAME, "seed": SEED, "fold": FOLD_IDX,
            "arch_hash": MODEL_HASH, "patch_size": list(PATCH_SIZE)}
    epoch = 0
    amp_enabled = (DEVICE.type == "cuda")
    loss_history = []  # per-epoch loss log
    first_epoch_loss = None

    for epoch in range(EPOCHS_BUDGET):
        if not budget_ok(MAX_TRAIN_HOURS):
            print(f"\n[TIME] Budget exhausted at epoch {epoch}. Stopping.")
            break

        # v2.2: Cosine + warmup LR
        lr = cosine_warmup_lr(epoch, EPOCHS_BUDGET, INITIAL_LR)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        # v2.3: Phase control + volume cycling (time-based)
        prev_k = dataset._current_k
        prev_ps = dataset.ps
        prev_phase = _PHASE
        dataset.set_epoch(epoch, elapsed_h())
        cur_k = dataset._current_k
        cur_ps = dataset.ps

        # v2.3: LR reduction at phase transition (prevent NaN cascade)
        if prev_phase == "learn" and _PHASE == "polish":
            lr = lr * 0.5
            for pg in optimizer.param_groups:
                pg["lr"] = lr
            print(f"  [TRANSITION] LR halved to {lr:.6f} at learn->polish boundary")
            # v2.5: Record phase change in meta
            if "phase_changes" not in meta:
                meta["phase_changes"] = []
            meta["phase_changes"].append({
                "epoch": epoch, "from": "learn", "to": "polish",
                "patch_size": list(cur_ps), "elapsed_h": round(elapsed_h(), 2)
            })

        # v2.5: Record K curriculum changes
        if cur_k != prev_k:
            if "k_changes" not in meta:
                meta["k_changes"] = []
            meta["k_changes"].append({
                "epoch": epoch, "k": cur_k, "elapsed_h": round(elapsed_h(), 2)
            })

        if epoch == 0 or cur_k != prev_k or cur_ps != prev_ps or prev_phase != _PHASE:
            pD, pH, pW = cur_ps
            oom_tag = f" OOM_LVL={_OOM_SHRINK}" if _OOM_SHRINK > 0 else ""
            print(f"  [PHASE={_PHASE}] patch={pD}x{pH}x{pW}, K={cur_k}, "
                  f"topo_scale={_TOPO_SCALE:.1f}, ds_scale={_DS_SCALE:.1f}, "
                  f"elastic={_ELASTIC_P:.2f}, medial={_MEDIAL_P:.1f}, "
                  f"lr={lr:.6f}, clip={GRAD_CLIP}, "
                  f"cache={dataset.vol_cache.cap}/{len(dataset.vols)} vols"
                  f"{oom_tag}", flush=True)

        model.train()
        ep_loss = 0.0
        ep_comps = {"ce": 0.0, "dice": 0.0, "msr": 0.0, "bnd": 0.0, "topo": 0.0}
        n_fwd = 0
        n_opt_steps = 0
        nan_count = 0
        t_ep = time.time()

        optimizer.zero_grad(set_to_none=True)

        for step_i, batch in enumerate(dataloader):
            img_b, lbl_b, med_b = batch
            img_b = img_b.to(DEVICE, non_blocking=True)
            lbl_b = lbl_b.to(DEVICE, non_blocking=True)
            med_b = med_b.to(DEVICE, non_blocking=True)

            try:
                with torch.amp.autocast("cuda", enabled=amp_enabled):
                    outputs = model(img_b)
                    if not isinstance(outputs, (list, tuple)):
                        outputs = [outputs]
                    loss, loss_comps = compute_loss(outputs, lbl_b, med_b)
                    loss_scaled = loss / GRAD_ACCUM

                # v2.4: NaN-safe step skipping
                # Skip ENTIRE step on NaN loss — don't backward, don't update EMA/optimizer
                if not torch.isfinite(loss):
                    optimizer.zero_grad(set_to_none=True)
                    scaler.update()  # Keep scaler state consistent
                    nan_count += 1
                    if nan_count % 3 == 0:
                        lr = max(lr * 0.5, LR_FLOOR)
                        for pg in optimizer.param_groups:
                            pg["lr"] = lr
                        print(f"  [NaN] {nan_count} NaN losses, LR backed off to {lr:.2e}")
                    continue

                scaler.scale(loss_scaled).backward()

                # v2.4: Check for NaN in gradients before optimizer step
                _has_nan_grad = False
                for p in model.parameters():
                    if p.grad is not None and not torch.isfinite(p.grad).all():
                        _has_nan_grad = True
                        break
                if _has_nan_grad:
                    optimizer.zero_grad(set_to_none=True)
                    scaler.update()
                    nan_count += 1
                    if nan_count % 3 == 0:
                        lr = max(lr * 0.5, LR_FLOOR)
                        for pg in optimizer.param_groups:
                            pg["lr"] = lr
                        print(f"  [NaN-grad] {nan_count} NaN grads, LR backed off to {lr:.2e}")
                    continue

                ep_loss += loss.item()
                n_fwd += 1
                for k in loss_comps:
                    ep_comps[k] += loss_comps[k]

                # v2.2: Gradient accumulation
                if (step_i + 1) % GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    ema.update(model)
                    n_opt_steps += 1

                # Hard mining: record high-loss coords
                if n_fwd > 10 and loss.item() > ep_loss / n_fwd * 1.5:
                    vi = random.randint(0, len(train_vols) - 1)
                    pD, pH, pW = PATCH_SIZE
                    dataset.add_hard_coord(vi, pD // 2, pH // 2, pW // 2)

                # Heartbeat: progress every 50 steps
                if (step_i + 1) % 50 == 0:
                    avg_so_far = ep_loss / max(n_fwd, 1)
                    el_ep = time.time() - t_ep
                    eta_ep = el_ep / (step_i + 1) * (ITERS_PER_EPOCH - step_i - 1)
                    print(f"    [{step_i+1}/{ITERS_PER_EPOCH}] loss={avg_so_far:.4f} | "
                          f"{el_ep:.0f}s elapsed | ETA {eta_ep:.0f}s", flush=True)

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    _OOM_THIS_EPOCH += 1
                    gc.collect()
                    torch.cuda.empty_cache()
                    optimizer.zero_grad(set_to_none=True)
                    # v2.5: OOM degradation — escalate if repeated
                    if _OOM_THIS_EPOCH >= 3 and _OOM_SHRINK < 3:
                        _OOM_SHRINK += 1
                        print(f"  [OOM-DEGRADE] Level {_OOM_SHRINK}: "
                              f"will shrink patch by {_OOM_SHRINK * 16} next epoch")
                    else:
                        print(f"  [OOM] Step {step_i}, count={_OOM_THIS_EPOCH}, clearing cache")
                    continue
                raise

        # Handle remaining accumulated gradients
        if (step_i + 1) % GRAD_ACCUM != 0 and n_fwd > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            ema.update(model)
            n_opt_steps += 1

        avg_loss = ep_loss / max(n_fwd, 1)
        dt = time.time() - t_ep
        remain_h = MAX_TRAIN_HOURS - elapsed_h()

        # v2.5: OOM recovery — if no OOMs this epoch and degraded, recover one level
        if _OOM_THIS_EPOCH == 0 and _OOM_SHRINK > 0:
            _OOM_SHRINK -= 1
            print(f"  [OOM-RECOVER] No OOM this epoch, shrink level -> {_OOM_SHRINK}")
        _OOM_THIS_EPOCH = 0

        # Per-component loss logging
        comp_str = " ".join(f"{k}={v/max(n_fwd,1):.4f}" for k, v in ep_comps.items() if v > 0)
        loss_history.append({"epoch": epoch, "loss": avg_loss, "phase": _PHASE,
                            "patch_size": list(dataset.ps), "lr": lr,
                            **{k: v/max(n_fwd,1) for k, v in ep_comps.items()}})

        # Smoke test after first epoch
        if epoch == 0:
            first_epoch_loss = avg_loss
            if avg_loss > 50.0:
                print(f"  [WARN] First epoch loss={avg_loss:.4f} is very high!")

        # Save best by loss (EMA weights, secondary)
        if avg_loss < best_loss:
            best_loss = avg_loss
            ema.save(best_loss_path)
            meta["best_loss_epoch"] = epoch
            meta["best_loss"] = best_loss

        # Val proxy for primary checkpoint (using EMA weights)
        val_str = ""
        if (epoch + 1) % VAL_EVERY == 0 and budget_ok(MAX_TRAIN_HOURS - 0.3):
            # Temporarily load EMA weights for validation
            saved_state = {k: v.clone() for k, v in model.state_dict().items()}
            ema.apply(model)
            vd, vc = val_dice_proxy(model)  # v2.5: returns (dice, combo)
            model.load_state_dict(saved_state)

            val_str = f" | val_dice={vd:.4f} combo={vc:.4f}"
            if vd > best_val_dice:
                best_val_dice = vd
                ema.save(best_val_path)
                meta["best_val_epoch"] = epoch
                meta["best_val_dice"] = best_val_dice
                val_str += " *dice*"
            # v2.5: Best-by-combo checkpoint (competition-metric-aligned)
            if vc > best_val_combo:
                best_val_combo = vc
                ema.save(best_combo_path)
                meta["best_combo_epoch"] = epoch
                meta["best_combo"] = best_val_combo
                val_str += " *combo*"

        # v2.5: Resource logging every 5 epochs
        if epoch % 5 == 0:
            log_resources(f"ep{epoch}")

        print(f"  Ep {epoch:3d}/{EPOCHS_BUDGET} | loss={avg_loss:.4f} | {comp_str} | "
              f"lr={lr:.6f} | opt={n_opt_steps} | {dt:.0f}s | rem={remain_h:.2f}h{val_str}")

        meta["last_epoch"] = epoch
        meta["last_loss"] = avg_loss
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)

    # If val never beat default, copy best_loss -> best
    if best_val_dice < 0 and os.path.exists(best_loss_path):
        shutil.copy2(best_loss_path, best_val_path)
        meta["best_val_epoch"] = meta.get("best_loss_epoch", 0)
        meta["best_val_dice"] = 0.0
        print("[WARN] No val improvement recorded; using best_loss as primary checkpoint")

    # v2.5: If combo never beat default, copy best_val -> best_combo
    if best_val_combo < 0 and os.path.exists(best_val_path):
        shutil.copy2(best_val_path, best_combo_path)
        meta["best_combo_epoch"] = meta.get("best_val_epoch", meta.get("best_loss_epoch", 0))
        meta["best_combo"] = 0.0

    # Save last (EMA weights)
    last_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_last.pt")
    ema.save(last_path)
    meta["total_epochs"] = epoch + 1
    meta["total_time_h"] = elapsed_h()

    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    # v2.5: Save history as separate file (keeps meta small, history easy to analyze)
    history_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_history.json")
    with open(history_path, "w") as f:
        json.dump(loss_history, f, indent=2)
    print(f"[HISTORY] Saved: {history_path}")

    log_resources("final")
    print(f"\n[DONE] {MODEL_NAME}: {epoch+1} epochs in {elapsed_h():.2f}h")
    print(f"  Best val dice: {best_val_dice:.4f} (epoch {meta.get('best_val_epoch', '?')})")
    print(f"  Best val combo: {best_val_combo:.4f} (epoch {meta.get('best_combo_epoch', '?')})")
    print(f"  Best loss: {best_loss:.5f} (epoch {meta.get('best_loss_epoch', '?')})")
    print(f"  Optimizer steps: {n_opt_steps} (last epoch)")
    print(f"  Checkpoints: best={best_val_path}, combo={best_combo_path}, last={last_path}")
    return best_val_path, meta

best_ckpt_path, train_meta = train_model()


In [ ]:

# ============================================================
# Threshold Calibration + Temperature Calibration + Enhanced Val Proxy
# ============================================================

@torch.no_grad()
def sliding_window_inference(vol_f32, model_eval, roi, overlap=0.25):
    """Sliding window inference with Gaussian weighting. Returns softmax probs (C,D,H,W)."""
    D, H, W = vol_f32.shape
    rD, rH, rW = roi
    sD = max(1, int(rD * (1.0 - overlap)))
    sH = max(1, int(rH * (1.0 - overlap)))
    sW = max(1, int(rW * (1.0 - overlap)))

    def _gauss_1d(n):
        if n <= 1: return np.ones(n, dtype=np.float32)
        x = np.linspace(-1, 1, n, dtype=np.float32)
        return np.exp(-2 * x * x)
    w3d = (_gauss_1d(rD)[:, None, None] *
           _gauss_1d(rH)[None, :, None] *
           _gauss_1d(rW)[None, None, :])
    w3d /= w3d.max() + 1e-8

    acc = np.zeros((NUM_CLASSES, D, H, W), dtype=np.float32)
    wacc = np.zeros((D, H, W), dtype=np.float32)

    z_starts = sorted(set(list(range(0, max(1, D-rD+1), sD)) + [max(0, D-rD)]))
    y_starts = sorted(set(list(range(0, max(1, H-rH+1), sH)) + [max(0, H-rH)]))
    x_starts = sorted(set(list(range(0, max(1, W-rW+1), sW)) + [max(0, W-rW)]))

    amp_on = (DEVICE.type == "cuda")
    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                patch = vol_f32[z0:z0+rD, y0:y0+rH, x0:x0+rW]
                if patch.shape != (rD, rH, rW):
                    continue
                t = torch.from_numpy(patch[None, None]).to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=amp_on):
                    logits = model_eval(t)
                    if isinstance(logits, (list, tuple)):
                        logits = logits[0]
                probs = torch.softmax(logits[0].float(), dim=0).cpu().numpy()
                for c in range(NUM_CLASSES):
                    acc[c, z0:z0+rD, y0:y0+rH, x0:x0+rW] += probs[c] * w3d
                wacc[z0:z0+rD, y0:y0+rH, x0:x0+rW] += w3d

    wacc = np.maximum(wacc, 1e-8)
    for c in range(NUM_CLASSES):
        acc[c] /= wacc
    return acc


def calibrate_temperature(cal_probs, cal_labels):
    """Find temperature that minimizes NLL on calibration data."""
    best_t = 1.0
    best_nll = float('inf')
    for t_cand in np.arange(0.5, 3.05, 0.1):
        nll = 0.0
        n = 0
        for fp, lbl in zip(cal_probs, cal_labels):
            valid = (lbl != IGNORE_LABEL) & (lbl != 2)
            if valid.sum() == 0:
                continue
            # Convert prob to logit, scale by temperature, convert back
            fp_clip = np.clip(fp[valid], 1e-7, 1 - 1e-7)
            logit = np.log(fp_clip / (1 - fp_clip)) / t_cand
            p_scaled = 1.0 / (1.0 + np.exp(-logit))
            gt = (lbl[valid] == 1).astype(np.float32)
            p_safe = np.clip(p_scaled, 1e-7, 1 - 1e-7)
            nll += -(gt * np.log(p_safe) + (1 - gt) * np.log(1 - p_safe)).sum()
            n += valid.sum()
        if n > 0:
            avg_nll = nll / n
            if avg_nll < best_nll:
                best_nll = avg_nll
                best_t = float(t_cand)
    return best_t


def enhanced_val_metrics(fg_prob, lbl, tl, th):
    """
    Compute enhanced validation metrics on a single volume.
    Returns dict with dice, comp_score, lcc_score, hole_score, bnd_f1, combined.
    """
    struct26 = generate_binary_structure(3, 3)
    gt = (lbl == 1)
    ignore = (lbl == 2) | (lbl == IGNORE_LABEL)

    # Apply hysteresis to get prediction
    strong = fg_prob >= th
    weak = fg_prob >= tl
    cc_w, n_w = cc_label(weak, structure=struct26)
    if n_w == 0:
        pred = np.zeros_like(fg_prob, dtype=bool)
    else:
        strong_ids = np.unique(cc_w[strong])
        strong_ids = strong_ids[strong_ids != 0]
        pred = np.isin(cc_w, strong_ids)

    pred[ignore] = False
    gt_clean = gt.copy()
    gt_clean[ignore] = False

    # 1. Dice
    inter = (pred & gt_clean).sum()
    denom = pred.sum() + gt_clean.sum()
    dice = float(2.0 * inter + 1) / float(denom + 1)

    # 2. Component count similarity
    cc_pred, n_pred = cc_label(pred, structure=struct26)
    cc_gt, n_gt = cc_label(gt_clean, structure=struct26)
    comp_score = max(0, 1.0 - abs(n_pred - n_gt) / max(n_gt, 5))

    # 3. Largest CC ratio
    if n_pred > 0 and pred.sum() > 0:
        sizes_pred = np.bincount(cc_pred.ravel())[1:]
        lcc_pred = sizes_pred.max() / pred.sum()
    else:
        lcc_pred = 0.0
    if n_gt > 0 and gt_clean.sum() > 0:
        sizes_gt = np.bincount(cc_gt.ravel())[1:]
        lcc_gt = sizes_gt.max() / gt_clean.sum()
    else:
        lcc_gt = 1.0
    lcc_score = max(0, 1.0 - abs(lcc_pred - lcc_gt))

    # 4. Hole proxy (BG holes inside prediction)
    struct2d = generate_binary_structure(2, 1)
    n_holes_pred = 0
    n_holes_gt = 0
    for z in range(pred.shape[0]):
        # Prediction holes
        bg_p = ~pred[z]
        lbl_p, n_p = cc_label(bg_p, structure=struct2d)
        border_ids_p = set()
        border_ids_p.update(lbl_p[0, :].tolist())
        border_ids_p.update(lbl_p[-1, :].tolist())
        border_ids_p.update(lbl_p[:, 0].tolist())
        border_ids_p.update(lbl_p[:, -1].tolist())
        n_holes_pred += sum(1 for k in range(1, n_p + 1) if k not in border_ids_p)
        # GT holes
        bg_g = ~gt_clean[z]
        lbl_g, n_g = cc_label(bg_g, structure=struct2d)
        border_ids_g = set()
        border_ids_g.update(lbl_g[0, :].tolist())
        border_ids_g.update(lbl_g[-1, :].tolist())
        border_ids_g.update(lbl_g[:, 0].tolist())
        border_ids_g.update(lbl_g[:, -1].tolist())
        n_holes_gt += sum(1 for k in range(1, n_g + 1) if k not in border_ids_g)
    hole_score = max(0, 1.0 - abs(n_holes_pred - n_holes_gt) / max(n_holes_gt, 10))

    # 5. Boundary F1
    bnd_pred = ndi.binary_dilation(pred, iterations=1) & ~pred
    bnd_gt = ndi.binary_dilation(gt_clean, iterations=1) & ~gt_clean
    bnd_inter = (bnd_pred & bnd_gt).sum()
    bnd_denom = bnd_pred.sum() + bnd_gt.sum()
    bnd_f1 = float(2.0 * bnd_inter + 1) / float(bnd_denom + 1)

    # v2.4: Combined proxy reweighted to HEAVILY penalize fragmentation
    # VOI (0.35 of LB) directly punishes component mismatch, so comp+lcc must dominate
    # Topo (0.30 of LB) punishes handles/tunnels, so hole_score matters
    # SurfaceDice (0.35 of LB) ~ bnd_f1
    combined = (0.15 * dice + 0.20 * bnd_f1 + 0.30 * comp_score +
                0.20 * lcc_score + 0.15 * hole_score)

    # v2.4: Hard penalty for extreme fragmentation (>5x gt components = disaster)
    if n_gt > 0 and n_pred > n_gt * 5:
        frag_penalty = min(0.5, (n_pred / max(n_gt, 1) - 5) * 0.05)
        combined = combined * (1.0 - frag_penalty)

    return {
        "dice": dice, "comp_score": comp_score, "lcc_score": lcc_score,
        "hole_score": hole_score, "bnd_f1": bnd_f1, "combined": combined,
        "n_pred_cc": n_pred, "n_gt_cc": n_gt,
    }


def calibrate_threshold():
    """Grid search for best hysteresis thresholds on val volumes."""
    model.eval()
    model.deep_supervision = False

    n_cal = min(VAL_N, len(val_vols))
    if n_cal == 0:
        print("[CAL] No val volumes, using defaults tl=0.34, th=0.62")
        return 0.34, 0.62, 0.85, 0.0

    best_score = -1
    best_tl, best_th = 0.34, 0.62

    cal_probs = []
    cal_labels = []
    for vi in range(n_cal):
        info = val_vols[vi]
        img = normalize_volume(read_tif(info["image"]))
        lbl = read_tif(info["label"]).astype(np.uint8)
        roi = tuple(min(p, s) for p, s in zip(PATCH_SIZE, img.shape))
        probs = sliding_window_inference(img, model, roi, overlap=VAL_OVERLAP)
        fg_prob = probs[1]
        cal_probs.append(fg_prob)
        cal_labels.append(lbl)
        print(f"  [CAL] Volume {info['id']}: prob range [{fg_prob.min():.3f}, {fg_prob.max():.3f}]")

    # Threshold grid search using leaderboard-aligned combined score
    # (not plain Dice — top competitors tune for the actual evaluation metric)
    for th in np.arange(0.50, 0.72, 0.02):
        for tl in np.arange(max(0.30, th - 0.25), th - 0.04, 0.02):
            scores = []
            for fp, lbl in zip(cal_probs, cal_labels):
                m = enhanced_val_metrics(fp, lbl, tl, th)
                scores.append(m["combined"])

            avg = np.mean(scores)
            if avg > best_score:
                best_score = avg
                best_tl, best_th = float(tl), float(th)

    print(f"[CAL] Best thresholds: tl={best_tl:.2f}, th={best_th:.2f}, combo_proxy={best_score:.4f}")

    # v2.2: Temperature calibration
    temperature = calibrate_temperature(cal_probs, cal_labels)
    print(f"[CAL] Temperature: {temperature:.2f}")

    # v2.2: Enhanced validation metrics at best thresholds
    print(f"\n[VAL] Enhanced validation metrics:")
    all_metrics = []
    for fp, lbl, info in zip(cal_probs, cal_labels, val_vols[:n_cal]):
        metrics = enhanced_val_metrics(fp, lbl, best_tl, best_th)
        all_metrics.append(metrics)
        print(f"  {info['id']}: dice={metrics['dice']:.4f}, comp={metrics['comp_score']:.3f}, "
              f"lcc={metrics['lcc_score']:.3f}, hole={metrics['hole_score']:.3f}, "
              f"bnd_f1={metrics['bnd_f1']:.3f}, combined={metrics['combined']:.4f}")
        print(f"    Components: pred={metrics['n_pred_cc']}, gt={metrics['n_gt_cc']}")

    avg_combined = np.mean([m["combined"] for m in all_metrics])
    print(f"  Average combined proxy: {avg_combined:.4f}")

    model.deep_supervision = True
    # v2.4b: Return combo_proxy so it can be stored in meta
    return best_tl, best_th, temperature, avg_combined


# v2.5: Prefer best_combo checkpoint (optimized for competition metric proxy)
combo_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best_combo.pt")
ema_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best.pt")
if os.path.exists(combo_path):
    ema_sd = torch.load(combo_path, map_location=DEVICE)
    model.load_state_dict(ema_sd)
    print(f"[CAL] Loaded best_combo checkpoint: {combo_path}")
elif os.path.exists(ema_path):
    ema_sd = torch.load(ema_path, map_location=DEVICE)
    model.load_state_dict(ema_sd)
    print(f"[CAL] Loaded best_dice checkpoint: {ema_path}")

# Run calibration
if budget_ok(MAX_TRAIN_HOURS + 0.5):
    cal_tl, cal_th, cal_temp, cal_combo = calibrate_threshold()
else:
    cal_tl, cal_th, cal_temp, cal_combo = 0.34, 0.62, 0.85, 0.0
    print("[CAL] Skipped (time budget), using researcher defaults")

# Update meta
meta_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_meta.json")
with open(meta_path, "r") as f:
    meta = json.load(f)
meta["tl"] = cal_tl
meta["th"] = cal_th
meta["temperature"] = cal_temp
meta["combined_proxy"] = cal_combo  # v2.4b: primary trust metric for ensemble
meta["model_role"] = MODEL_ROLE
# v2.4: Per-model dust size (connectivity models get stronger dust removal)
if MODEL_ROLE == "connectivity":
    meta["dust_min"] = 512  # 2.7x global default of 192
    meta["inf_overlap"] = 0.625  # Higher overlap for small-patch model (reduce stitching artifacts)
elif MODEL_ROLE == "structure":
    meta["dust_min"] = 128  # Slightly lighter (preserve recall)
    meta["inf_overlap"] = 0.5
else:
    meta["dust_min"] = 192  # Default
    meta["inf_overlap"] = 0.5

# v2.5: Copy calibrated checkpoint to primary path for prediction notebook
# The combo checkpoint was used for calibration; ensure model_X_best.pt exists
primary_path = os.path.join(CKPT_DIR, f"{MODEL_NAME}_best.pt")
if os.path.exists(combo_path) and combo_path != primary_path:
    shutil.copy2(combo_path, primary_path)
    print(f"[CAL] Copied calibrated checkpoint -> {primary_path}")

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n[FINAL] {MODEL_NAME} artifacts in {CKPT_DIR}:")
for fn in sorted(os.listdir(CKPT_DIR)):
    if MODEL_NAME in fn:
        sz = os.path.getsize(os.path.join(CKPT_DIR, fn))
        print(f"  {fn}: {sz/1e6:.1f} MB")
